# 🛡️ PharmaGuard — AI Pharmacy Security Monitor
### Real-time Suspicious-Interaction Monitoring System

This Google Colab notebook runs the cloud processing backend for **PharmaGuard**.
It ingests live webcam frames transmitted securely from your laptop browser, runs GPU-accelerated person detection (pretrained YOLO) and multi-object tracking (ByteTrack), analyzes pharmacy shelf dwell interactions, and streams annotated telemetry back to your browser in real time.

---

## Step 1: Verify GPU Environment
Check that a GPU (T4, V100, A100, or L4) is allocated to this Colab runtime.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. For real-time performance, select Runtime > Change runtime type > T4 GPU.")

## Step 2: Mount Google Drive (Optional for Persistent Model & Alert Storage)
Persisting weights on Google Drive avoids re-downloading models if your Colab session restarts.

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

PERSISTENT_DIR = "/content/drive/MyDrive/PharmaGuard_Artifacts"
os.makedirs(PERSISTENT_DIR, exist_ok=True)
print(f"Google Drive persistent storage ready at: {PERSISTENT_DIR}")

## Step 3: Clone or Sync PharmaGuard Repository

In [ ]:
# Set your repository URL
REPO_URL = "https://github.com/your-username/pharmguard.git"

import os
if not os.path.exists("/content/pharmguard"):
    !git clone $REPO_URL /content/pharmguard
else:
    %cd /content/pharmguard
    !git pull

%cd /content/pharmguard

## Step 4: Install Pinned Dependencies

In [ ]:
!pip install -q -r requirements.txt

## Step 5: Verify Model Weights & Cache
Warm up YOLOv8n weights and verify inference on a dummy test frame.

In [ ]:
import numpy as np
from pharmaguard.detector import YOLODetector, DetectorConfig

print("Loading YOLO detector on GPU...")
detector = YOLODetector(DetectorConfig(model_name="yolov8n.pt", device="cuda" if torch.cuda.is_available() else "cpu"))

# Run warmup test inference
dummy_frame = np.zeros((480, 640, 3), dtype=np.uint8)
result = detector.detect(dummy_frame)
print("Detector loaded and verified successfully! Model metadata:", detector.get_model_info())

## Step 6: Launch PharmaGuard Cloud Server with Public Gradio Link
This starts the surveillance backend and outputs a public **Gradio Live URL** (e.g. `https://xxxx.gradio.live`).

**How to connect your laptop webcam:**
1. Click the public `https://xxxx.gradio.live` link generated below.
2. On your laptop browser, click the **Webcam** icon on the left panel.
3. When the browser prompts `Allow access to your camera?`, click **Allow**.
4. The browser will begin streaming frames to this Colab GPU runtime, and you will see the real-time AI bounding boxes, tracking IDs, shelf dwell monitoring, and alerts rendered live!

In [ ]:
# Run PharmaGuard with Colab configuration
!python main.py --colab --model yolov8n.pt